# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrishaSolanki-coder/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../src"))
import pipeline as pl

df = pl.load_data("../data/raw/content_refresh_anonymized.csv")
df = pl.build_temporal_features_and_label(df)     # engineers *_hist60, ctr_hist60, label
df = pl.clean_and_impute(df)                       # content_type-conditional median fill

X, y = pl.prepare_matrix(df, pl.NUMERIC_FEATURE_COLS)   # one-hot encodes categoricals
print(X.shape)
X.head(3)


## 2. Feature notes (meaning, missing, categorical, available-when?)

Feature	Meaning	Missing?	Available before cutoff?
impressions_hist60 / clicks_hist60 / sessions_hist60	derived from the 60 days before the label window (oldest_30d + prev_30d)	no NaNs (computed, always ≥0)	✅ yes
ctr_hist60	clicks/impressions over the same 60d, ×100	0 when impressions_hist60=0	✅ yes
search_volume, cpc	keyword-level market signals	100% missing for feedly article — content_type-conditional median fill	✅ yes (static)
word_count, char_count	content attributes	~28% missing for keyword article — same conditional fill	✅ yes (static)
content_age_days, days_since_last_update, age_tier_order	page age/freshness at snapshot	none	✅ yes
content_type, main_intent, provider_used, model_used, competition_level, age_tier, freshness_tier, word_count_tier, char_count_tier	categorical, one-hot encoded (dummy_na=True)	NA becomes its own dummy column	✅ yes (static)

In [2]:
import pandas as pd

# Load FlyRank dataset
url = "https://raw.githubusercontent.com/PrishaSolanki-coder/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

# Numeric features used in the Content Refresh project
NUMERIC_FEATURE_COLS = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

# Check missingness by content type
missingness_by_content_type = (
    df.groupby("content_type")[NUMERIC_FEATURE_COLS]
    .apply(lambda x: x.isna().mean() * 100)
    .round(2)
)

print("Dataset shape:", df.shape)
print("\nMissingness by content type (%):")
missingness_by_content_type


Dataset shape: (30000, 44)

Missingness by content type (%):


,search_volume,competition,cpc,word_count,impressions_90d,clicks_90d,sessions_90d,days_since_last_update,ctr,avg_position
content_type,,,,,,,,,,
comparison article,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0
feedly article,100.00,100.00,100.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0
keyword article,1.37,1.37,1.37,28.3,0.0,0.0,0.0,0.0,0.0,0.0


## 3. The leakage hunt
Attack: does any feature let a model reconstruct the label directly? Checked column membership + correlation. Earlier version failed this test — impressions_last_30d/prev_30d are the exact inputs to the trend fields and let a model hit AUC ≈0.997. Fixed by folding them into *_hist60 only and dropping every raw last/prev column and every 90d-only aggregate from features.

In [3]:
import pandas as pd

# Load the FlyRank dataset
url = "https://raw.githubusercontent.com/PrishaSolanki-coder/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

# Feature definitions
NUMERIC_FEATURE_COLS = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

CATEGORICAL_FEATURE_COLS = [
    "content_type",
    "main_intent",
    "provider_used",
    "model_used"
]

LABEL_COL = "is_declining_label"

# Create label if it does not already exist
if LABEL_COL not in df.columns:
    df[LABEL_COL] = (df["trend_direction"] == "down").astype(int)


# ------------------------------------------------------------
# 1. COLUMN LEAKAGE CHECK
# ------------------------------------------------------------

BANNED_COLUMNS = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id"
]

FEATURE_COLS = NUMERIC_FEATURE_COLS + CATEGORICAL_FEATURE_COLS

leaked_columns = [
    col for col in FEATURE_COLS
    if col in BANNED_COLUMNS
]

if leaked_columns:
    raise ValueError(
        f"Leakage detected! Banned columns found in features: {leaked_columns}"
    )

print("Column check: passed")


# ------------------------------------------------------------
# 2. CORRELATION CHECK WITH LABEL
# ------------------------------------------------------------

corr = (
    df[NUMERIC_FEATURE_COLS + [LABEL_COL]]
    .corr(numeric_only=True)[LABEL_COL]
    .drop(LABEL_COL)
    .sort_values(key=lambda x: x.abs(), ascending=False)
)

print(
    "Max |correlation| with label:",
    round(corr.abs().max(), 4)
)

print("\nTop 5 numeric feature correlations with label:")
print(corr.head(5))


Column check: passed
Max |correlation| with label: 0.0902

Top 5 numeric feature correlations with label:
word_count                0.090157
days_since_last_update    0.081383
ctr                      -0.061911
clicks_90d               -0.039680
avg_position             -0.029035
Name: is_declining_label, dtype: float64


## 4. What I excluded and why

trend_direction, trend_pct          -> the original label's source fields
is_declining_label                  -> old label, never a feature
impressions/clicks/sessions_last_30d -> IS the label window itself
impressions/clicks/sessions_prev_30d -> folded into *_hist60 instead, not used raw
impressions_90d, clicks_90d, sessions_90d -> blend in the last_30d label period
ctr, avg_position, engagement_rate,
scroll_rate, ai_traffic_pct,
pageviews_90d, users_90d,
engaged_sessions_90d, ai_sessions_90d,
scroll_events_90d, days_with_*,
impression_tier, position_tier      -> all 90d-only aggregates with no last/prev
                                        split available, so can't be cleanly
                                        separated from the label window
content_id, client_id               -> identifiers; used only for joins/grouping

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.